# CSE-CIC-IDS2018 Preprocessing Notebook

This notebook is used for Stage 1 data exploration and preprocessing experiments.

The final reusable preprocessing logic should be kept in:

`stage-1/scripts/sample_cse_cic_ids2018.py`

Large raw CSV files should not be committed to GitHub.

## Archive inspection note

The local `archive.zip` was inspected before this notebook was updated. It contains 10 CSV files named by date, such as `02-14-2018.csv`, `02-20-2018.csv`, and `03-02-2018.csv`.

Most files contain 80 columns starting with `Dst Port`, `Protocol`, and `Timestamp`, ending with `Label`. The `02-20-2018.csv` file contains 84 columns and additionally includes `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`. Because of this mixed schema, the notebook uses safe column access and fallback values when source or destination IP fields are missing.

## 1. Setup

In [ ]:
# Colab option A: upload archive.zip manually.
# from google.colab import files
# uploaded = files.upload()

# Colab option B: mount Google Drive if archive.zip is stored there.
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd

# Update this path in Colab after uploading or mounting the dataset archive.
ARCHIVE_PATH = Path('/content/archive.zip')
OUTPUT_PATH = Path('/content/sample-alerts.json')

RANDOM_STATE = 42

## 2. Load Dataset

In [ ]:
# Inspect the archive without extracting large CSV files.
# Expected from the inspected local archive: 10 CSV files.
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    csv_files = [name for name in archive.namelist() if name.lower().endswith('.csv')]

csv_files

In [ ]:
# Load one or more files for Stage 1 exploration.
# Start small. 02-20-2018.csv is very large and has extra IP columns.
SELECTED_FILES = [
    '02-16-2018.csv',
    # '02-20-2018.csv',
]

def read_csv_from_zip(archive_path, member_name, nrows=None):
    with zipfile.ZipFile(archive_path) as archive:
        with archive.open(member_name) as file:
            return pd.read_csv(file, nrows=nrows)

# Use nrows while exploring to avoid loading very large files into memory.
frames = [read_csv_from_zip(ARCHIVE_PATH, name, nrows=100_000) for name in SELECTED_FILES]
df = pd.concat(frames, ignore_index=True)
df.head()

## 3. Inspect Columns

In [ ]:
print('Rows, columns:', df.shape)
print(df.columns.tolist())
df.info()

In [ ]:
# Inspect column differences across CSV files in the archive.
schema_summary = []
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    for name in csv_files:
        preview = pd.read_csv(archive.open(name), nrows=0)
        columns = preview.columns.tolist()
        schema_summary.append({
            'file': name,
            'column_count': len(columns),
            'has_src_ip': 'Src IP' in columns,
            'has_dst_ip': 'Dst IP' in columns,
            'has_label': 'Label' in columns,
            'first_columns': columns[:8],
        })

pd.DataFrame(schema_summary)

## 4. Clean Data

In [ ]:
def clean_column_names(dataframe):
    cleaned = dataframe.copy()
    cleaned.columns = cleaned.columns.str.strip()
    return cleaned

def clean_invalid_values(dataframe):
    cleaned = dataframe.copy()
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
    cleaned = cleaned.dropna(subset=['Label'])
    return cleaned

df = clean_column_names(df)
df = clean_invalid_values(df)
df.head()

## 5. Check Label Distribution

In [ ]:
label_counts = df['Label'].value_counts(dropna=False)
label_counts

## 6. Sample Benign and Attack Rows

In [ ]:
def sample_benign_and_attack_rows(dataframe, benign_n=50, attack_n=50):
    labels = dataframe['Label'].astype(str)
    benign_rows = dataframe[labels.str.lower() == 'benign']
    attack_rows = dataframe[labels.str.lower() != 'benign']

    benign_sample = benign_rows.sample(
        n=min(benign_n, len(benign_rows)), random_state=RANDOM_STATE
    )
    attack_sample = attack_rows.sample(
        n=min(attack_n, len(attack_rows)), random_state=RANDOM_STATE
    )

    return pd.concat([benign_sample, attack_sample], ignore_index=True)

sample_df = sample_benign_and_attack_rows(df, benign_n=5, attack_n=5)
sample_df['Label'].value_counts(dropna=False)

## 7. Convert Rows to Dashboard Alerts

In [ ]:
PROTOCOL_MAP = {
    0: 'HOPOPT',
    1: 'ICMP',
    6: 'TCP',
    17: 'UDP',
}

def safe_value(row, field, default='unknown'):
    if field not in row.index:
        return default
    value = row[field]
    if pd.isna(value):
        return default
    return value

def protocol_name(value):
    try:
        return PROTOCOL_MAP.get(int(value), str(value))
    except (TypeError, ValueError):
        return str(value)

def severity_for_label(label):
    normalized = label.lower()
    if normalized == 'benign':
        return 'Low'
    if 'ddos' in normalized or 'dos' in normalized or 'bot' in normalized:
        return 'High'
    if 'brute' in normalized or 'infilteration' in normalized or 'infiltration' in normalized:
        return 'High'
    return 'Medium'

def risk_for_label(label):
    return 24 if label.lower() == 'benign' else 82

def row_to_alert(row, index):
    label = str(safe_value(row, 'Label', 'Benign'))
    is_benign = label.lower() == 'benign'
    risk_score = risk_for_label(label)
    source_ip = str(safe_value(row, 'Src IP', 'unknown-source'))
    destination_ip = str(safe_value(row, 'Dst IP', 'unknown-destination'))
    port = int(float(safe_value(row, 'Dst Port', 0)))
    protocol = protocol_name(safe_value(row, 'Protocol', 'TCP'))

    return {
        'id': f'AL-{index:04d}',
        'timestamp': str(safe_value(row, 'Timestamp', 'unknown-time')),
        'sourceIp': source_ip,
        'destinationIp': destination_ip,
        'protocol': protocol,
        'port': port,
        'attackType': 'Benign' if is_benign else label,
        'severity': severity_for_label(label),
        'confidence': 72 if is_benign else 86,
        'baseRiskScore': risk_score,
        'currentRiskScore': risk_score,
        'status': 'Unreviewed',
        'groundTruth': 'benign' if is_benign else 'malicious',
        'similarityKey': 'benign' if is_benign else label.lower().replace(' ', '-'),
        'triggerReason': 'Flow pattern matches a labelled CSE-CIC-IDS2018 scenario.',
        'evidence': f"Dataset label: {label}; destination port: {port}; protocol: {protocol}.",
        'recommendedAction': 'Review related traffic and confirm whether escalation is needed.',
    }

alerts = [row_to_alert(row, i + 1) for i, (_, row) in enumerate(sample_df.iterrows())]
alerts[:2]

## 8. Validate Alert Schema

In [ ]:
required_fields = [
    'id', 'timestamp', 'sourceIp', 'destinationIp', 'protocol', 'port',
    'attackType', 'severity', 'confidence', 'baseRiskScore',
    'currentRiskScore', 'status', 'groundTruth', 'similarityKey',
    'triggerReason', 'evidence', 'recommendedAction'
]

for alert in alerts:
    missing = [field for field in required_fields if field not in alert]
    assert not missing, f"{alert.get('id')} missing fields: {missing}"
    assert alert['groundTruth'] in {'malicious', 'benign'}
    assert alert['status'] == 'Unreviewed'

print(f'Validated {len(alerts)} alert objects.')

## 9. Export sample-alerts.json

In [ ]:
with OUTPUT_PATH.open('w', encoding='utf-8') as file:
    json.dump(alerts, file, indent=2)

OUTPUT_PATH

## 10. Notes and Findings

Record dataset files used, sample size, label distribution, cleaning decisions, dropped or renamed columns, and the final output path here.

Initial local archive findings:

- `archive.zip` contains 10 CSV files.
- Most CSV files have 80 columns and do not include `Src IP` or `Dst IP`.
- `02-20-2018.csv` has 84 columns and includes `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`.
- All inspected files include `Label` as the final column.